# Collaborative Filtering

Training and evaluating 4 models:
- User-based CF
- Item-based CF
- SVD
- NMF

In [13]:
import sys
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')
from src.utils.data_loader import load_movielens, to_surprise_dataset
from src.utils.evaluator import rmse, mae, precision_recall_at_k
from src.collaborative.matrix_factorization import train_svd, train_nmf
from src.collaborative.user_based import UserBasedRecommender 
from src.collaborative.item_based import ItemBasedRecommender
from surprise.model_selection import train_test_split

os.makedirs('../models', exist_ok=True)
sns.set_style('whitegrid')

ImportError: cannot import name 'UserBasedRecommender' from 'src.collaborative.user_based' (d:\Infyntrk\movie-recommender-ml\notebooks\..\src\collaborative\user_based.py)

## 1. Load Data

In [ ]:
ratings_df, movies_df = load_movielens(
    ratings_path='../data/raw/u.data',
    movies_path='../data/raw/u.item'
)

data = to_surprise_dataset(ratings_df)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
print('Data loaded. Testset size:', len(testset))

## 2. User-Based Collaborative Filtering

In [ ]:
user_cf = UserBasedRecommender(k=20)
user_cf.fit(ratings_df)

print('User-item matrix shape:', user_cf.user_item_matrix.shape)
print('Sample recommendations for User 1:')
recs = user_cf.recommend(user_id=1, n=10)
print(recs)

## 3. Item-Based Collaborative Filtering

In [ ]:
item_cf = ItemBasedRecommender(k=20)
item_cf.fit(ratings_df)

print('Item similarity matrix shape:', item_cf.item_similarity.shape)
print('Sample recommendations for User 1:')
recs = item_cf.recommend(user_id=1, n=10)
print(recs)

## 4. SVD Model

In [ ]:
print('Training SVD...')
svd_model, svd_preds = train_svd(data)
svd_rmse = rmse(svd_preds)
svd_mae  = mae(svd_preds)
svd_p, svd_r = precision_recall_at_k(svd_preds, k=10)
print(f'SVD — RMSE: {svd_rmse:.4f}, MAE: {svd_mae:.4f}, P@10: {svd_p:.4f}')

joblib.dump(svd_model, '../models/svd_model.pkl')
print('svd_model.pkl saved!')

## 5. NMF Model

In [ ]:
print('Training NMF...')
nmf_model, nmf_preds = train_nmf(data)
nmf_rmse = rmse(nmf_preds)
nmf_mae  = mae(nmf_preds)
nmf_p, nmf_r = precision_recall_at_k(nmf_preds, k=10)
print(f'NMF — RMSE: {nmf_rmse:.4f}, MAE: {nmf_mae:.4f}, P@10: {nmf_p:.4f}')

## 6. Model Comparison Table

In [ ]:
results = pd.DataFrame({
    'Model': ['SVD', 'NMF'],
    'RMSE':  [svd_rmse, nmf_rmse],
    'MAE':   [svd_mae,  nmf_mae],
    'Precision@10': [svd_p, nmf_p],
})
results[['RMSE','MAE','Precision@10']] = results[['RMSE','MAE','Precision@10']].round(4)
print('=== Model Comparison ===')
display(results)

## 7. RMSE Comparison Chart

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(results['Model'], results['RMSE'], color=['steelblue', 'coral'], edgecolor='white')
plt.title('RMSE — SVD vs NMF', fontsize=13, fontweight='bold')
plt.ylabel('RMSE (lower is better)')
plt.tight_layout()
plt.savefig('../reports/figures/svd_nmf_rmse.png', dpi=150)
plt.show()
print('Saved: svd_nmf_rmse.png')